# SPARL: Spatial Proteomics Representation Learning

This tutorial demonstrates how to use SPARL for analyzing spatial proteomics data.

SPARL learns spatial-aware latent representations from protein expression data measured by technologies like imaging mass cytometry (IMC) or CODEX.

## Key Features:
- Spatial-aware representation learning
- Designed for protein expression data
- Supports dimensionality reduction
- Protein imputation capabilities

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd

# Import spatialvi
import spatialvi
from spatialvi.external import SPARL

sc.set_figure_params(figsize=(6, 6))
print("spatialvi version:", spatialvi.__version__)

## 1. Create Synthetic Spatial Proteomics Data

For this tutorial, we'll create synthetic data that mimics CODEX/IMC protein measurements.

In [ ]:
# Create synthetic spatial proteomics data
np.random.seed(42)

n_cells = 2000
n_proteins = 40  # Typical for IMC/CODEX panels

# Create cell types with different protein expression profiles
cell_types = ["T cells", "B cells", "Macrophages", "Epithelial", "Stromal"]
n_per_type = n_cells // len(cell_types)

# Protein names (common markers)
protein_names = [
    "CD3",
    "CD4",
    "CD8",
    "CD45",
    "CD45RO",
    "CD45RA",
    "CD19",
    "CD20",
    "CD79a",
    "PAX5",
    "CD68",
    "CD163",
    "CD14",
    "CD11b",
    "CD11c",
    "E-cadherin",
    "Pan-CK",
    "EpCAM",
    "HER2",
    "Ki67",
    "Vimentin",
    "SMA",
    "Collagen",
    "Fibronectin",
    "FAP",
    "PD-1",
    "PD-L1",
    "CTLA-4",
    "LAG-3",
    "TIM-3",
    "HLA-DR",
    "CD86",
    "CD80",
    "ICOS",
    "Granzyme B",
    "Perforin",
    "IFNg",
    "TNFa",
    "IL-6",
    "DAPI",
]

# Create expression profiles
X = np.zeros((n_cells, n_proteins))
labels = []

# Cell type-specific expression patterns
profiles = {
    "T cells": [0, 1, 2, 3, 25, 26, 34, 35],  # CD3, CD4, CD8, etc.
    "B cells": [6, 7, 8, 9, 3],  # CD19, CD20, etc.
    "Macrophages": [10, 11, 12, 13, 30],  # CD68, CD163, etc.
    "Epithelial": [15, 16, 17, 18, 19],  # E-cadherin, Pan-CK, etc.
    "Stromal": [20, 21, 22, 23, 24],  # Vimentin, SMA, etc.
}

for i, ct in enumerate(cell_types):
    start_idx = i * n_per_type
    end_idx = start_idx + n_per_type

    # Base expression (low)
    X[start_idx:end_idx] = np.random.exponential(0.5, (n_per_type, n_proteins))

    # Marker expression (high)
    for marker_idx in profiles[ct]:
        X[start_idx:end_idx, marker_idx] += np.random.exponential(3, n_per_type)

    labels.extend([ct] * n_per_type)

# Create spatial coordinates (simulate tissue structure)
# Cells of same type tend to cluster together
coords = np.zeros((n_cells, 2))
centers = {
    "T cells": (0.3, 0.5),
    "B cells": (0.7, 0.5),
    "Macrophages": (0.5, 0.3),
    "Epithelial": (0.5, 0.7),
    "Stromal": (0.5, 0.5),
}

for i, ct in enumerate(cell_types):
    start_idx = i * n_per_type
    end_idx = start_idx + n_per_type
    center = centers[ct]
    coords[start_idx:end_idx, 0] = np.random.normal(center[0], 0.15, n_per_type)
    coords[start_idx:end_idx, 1] = np.random.normal(center[1], 0.15, n_per_type)

# Create AnnData object
adata = sc.AnnData(X)
adata.var_names = protein_names
adata.obs["cell_type"] = pd.Categorical(labels)
adata.obsm["spatial"] = coords * 1000  # Scale to micrometers

print(adata)
print("\nCell type distribution:")
print(adata.obs["cell_type"].value_counts())

In [ ]:
# Visualize the spatial distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color by cell type
for ct in cell_types:
    mask = adata.obs["cell_type"] == ct
    axes[0].scatter(adata.obsm["spatial"][mask, 0], adata.obsm["spatial"][mask, 1], label=ct, alpha=0.6, s=10)
axes[0].legend()
axes[0].set_xlabel("X (um)")
axes[0].set_ylabel("Y (um)")
axes[0].set_title("Spatial Distribution by Cell Type")

# Heatmap of protein expression
sc.pl.heatmap(adata, var_names=protein_names[:15], groupby="cell_type", ax=axes[1], show=False)

plt.tight_layout()
plt.show()

## 2. Initialize SPARL Model

In [ ]:
# Initialize SPARL model
model = SPARL(
    adata,
    spatial_key="spatial",
    layer=None,  # Use adata.X
    n_latent=20,  # Latent space dimensions
)

print("SPARL model initialized")
print(f"Input features: {adata.n_vars} proteins")
print(f"Latent dimensions: {model.n_latent}")

## 3. Train the Model

Note: Training requires the `sparl` package to be installed.

In [ ]:
# Train the model
try:
    model.train(
        max_epochs=100,
        batch_size=128,
        lr=1e-3,
    )
    print("Training complete!")

except ImportError as e:
    print(f"Note: {e}")
    print("Install with: pip install sparl")
    print("Continuing with simulated embeddings for demonstration...")

    # Create simulated SPARL embeddings using PCA as approximation
    sc.pp.scale(adata)
    sc.tl.pca(adata, n_comps=model.n_latent)
    adata.obsm["X_sparl"] = adata.obsm["X_pca"]

## 4. Get Latent Representations

In [ ]:
# Get latent representations
try:
    latent = model.get_latent_representation()
    adata.obsm["X_sparl"] = latent
except (RuntimeError, AttributeError):
    print("Using pre-computed embeddings")
    latent = adata.obsm["X_sparl"]

print(f"Latent representation shape: {latent.shape}")

## 5. Visualize Latent Space

In [ ]:
# Compute UMAP on SPARL latent space
sc.pp.neighbors(adata, use_rep="X_sparl", n_neighbors=15)
sc.tl.umap(adata)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(adata, color="cell_type", ax=axes[0], show=False, title="SPARL UMAP - Cell Types")

# Spatial plot colored by UMAP coordinates
sc.pl.embedding(adata, basis="spatial", color="cell_type", ax=axes[1], show=False, title="Spatial - Cell Types")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize first few latent dimensions spatially
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i in range(6):
    adata.obs[f"sparl_dim_{i}"] = adata.obsm["X_sparl"][:, i]

    # Manual spatial plot
    scatter = axes[i].scatter(
        adata.obsm["spatial"][:, 0],
        adata.obsm["spatial"][:, 1],
        c=adata.obs[f"sparl_dim_{i}"],
        cmap="viridis",
        s=5,
        alpha=0.7,
    )
    plt.colorbar(scatter, ax=axes[i])
    axes[i].set_title(f"SPARL Dimension {i}")
    axes[i].set_xlabel("X (um)")
    axes[i].set_ylabel("Y (um)")

plt.tight_layout()
plt.show()

## 6. Cluster Analysis

In [ ]:
# Cluster based on SPARL embeddings
sc.tl.leiden(adata, resolution=0.5, key_added="sparl_cluster")

print("Cluster distribution:")
print(adata.obs["sparl_cluster"].value_counts())

In [ ]:
# Visualize clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(adata, color="sparl_cluster", ax=axes[0], show=False, title="SPARL Clusters (UMAP)")

# Spatial view
for cluster in adata.obs["sparl_cluster"].cat.categories:
    mask = adata.obs["sparl_cluster"] == cluster
    axes[1].scatter(
        adata.obsm["spatial"][mask, 0], adata.obsm["spatial"][mask, 1], label=f"Cluster {cluster}", alpha=0.6, s=10
    )
axes[1].legend()
axes[1].set_xlabel("X (um)")
axes[1].set_ylabel("Y (um)")
axes[1].set_title("SPARL Clusters (Spatial)")

plt.tight_layout()
plt.show()

In [ ]:
# Compare clusters with ground truth cell types
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Convert to numeric
true_labels = adata.obs["cell_type"].cat.codes
cluster_labels = adata.obs["sparl_cluster"].astype(int)

ari = adjusted_rand_score(true_labels, cluster_labels)
nmi = normalized_mutual_info_score(true_labels, cluster_labels)

print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Normalized Mutual Information: {nmi:.4f}")

## 7. Protein Marker Analysis

In [ ]:
# Find marker proteins for each cluster
sc.tl.rank_genes_groups(adata, "sparl_cluster", method="wilcoxon")

# Display top markers
sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False)

In [ ]:
# Visualize key markers spatially
key_markers = ["CD3", "CD19", "CD68", "Pan-CK", "Vimentin"]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, marker in enumerate(key_markers):
    expr = adata[:, marker].X.flatten()
    scatter = axes[i].scatter(
        adata.obsm["spatial"][:, 0], adata.obsm["spatial"][:, 1], c=expr, cmap="Reds", s=5, alpha=0.7
    )
    plt.colorbar(scatter, ax=axes[i])
    axes[i].set_title(marker)
    axes[i].set_xlabel("X (um)")
    axes[i].set_ylabel("Y (um)")

plt.tight_layout()
plt.show()

## 8. Protein Imputation (Optional)

In [ ]:
# Try protein imputation
try:
    # Impute proteins
    adata = model.impute_proteins(adata=adata, store_layer="sparl_imputed")

    print("Protein imputation complete!")
    print("Imputed layer: sparl_imputed")

    # Compare original vs imputed
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(adata.X[:, 0].flatten(), adata.layers["sparl_imputed"][:, 0].flatten(), alpha=0.3)
    axes[0].set_xlabel("Original")
    axes[0].set_ylabel("Imputed")
    axes[0].set_title(f"{protein_names[0]}: Original vs Imputed")

    # Correlation
    from scipy.stats import pearsonr

    for i, name in enumerate(protein_names[:5]):
        corr, _ = pearsonr(adata.X[:, i].flatten(), adata.layers["sparl_imputed"][:, i].flatten())
        print(f"  {name}: r = {corr:.4f}")

except (RuntimeError, AttributeError) as e:
    print(f"Imputation not available: {e}")

## 9. Neighborhood Analysis

In [ ]:
import seaborn as sns

# Analyze cell type neighborhoods
from sklearn.neighbors import NearestNeighbors

# Find spatial neighbors
coords = adata.obsm["spatial"]
nn = NearestNeighbors(n_neighbors=10)
nn.fit(coords)
distances, indices = nn.kneighbors(coords)

# Compute cell type co-localization
cell_type_labels = adata.obs["cell_type"].values
n_types = len(cell_types)

colocalization = np.zeros((n_types, n_types))

for i in range(len(adata)):
    ct_i = cell_types.index(cell_type_labels[i])
    for j in indices[i, 1:]:  # Exclude self
        ct_j = cell_types.index(cell_type_labels[j])
        colocalization[ct_i, ct_j] += 1

# Normalize
colocalization = colocalization / colocalization.sum(axis=1, keepdims=True)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(colocalization, xticklabels=cell_types, yticklabels=cell_types, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Cell Type Co-localization Matrix")
plt.xlabel("Neighbor Cell Type")
plt.ylabel("Center Cell Type")
plt.tight_layout()
plt.show()

## Summary

In this tutorial, we demonstrated:

1. How to prepare spatial proteomics data for SPARL
2. How to initialize and train the SPARL model
3. How to extract spatial-aware latent representations
4. How to visualize latent space and spatial patterns
5. How to perform clustering analysis
6. How to identify marker proteins
7. How to analyze cell type neighborhoods

SPARL provides a powerful framework for analyzing spatial proteomics data by learning representations that capture both protein expression patterns and spatial context.